# [확장 1 · 2026-10-26] 정책 안의 종목 순위 신호를 분리해서 평가 (1일 주기)

**단서.** 1일 주기 재학습(`crypto_horizon_retrain_1d`)에서 β=−90 전문가의 종목 점수(확신도 로짓)와 다음 날 수익률의 순위 상관(IC)이
검증 0.058(t=4.0)·테스트 0.042(t=4.1)로 **두 구간 모두 유의**했다. 그런데 정책은 노출 7%·스무딩 때문에 이 순위 정보를 거의 쓰지 않았다.

**가설.** 강화학습 전문가는 쓸 만한 종목 선택 신호를 학습했지만, 포트폴리오 구성(낮은 노출 게이트·스무딩·라우터)이 그 신호를 버리고 있다.

**반드시 함께 볼 대조군.** 단순 팩터(전일 수익률, 1주 수익률 — 각각 모멘텀·반전 방향, 저변동성). 단순 팩터가 같은 IC를 내면
그 신호는 RL이 배운 것이 아니라 흔한 반전·모멘텀 효과다. 두 신호의 순위 상관도 함께 기록한다.

## 사전 등록 (실행 전에 고정 — 결과를 보고 바꾸지 않는다)
| 항목 | 값 |
|---|---|
| 신호 원천 | 1일 주기 β 풀 4개 (학습 없음, `models/hz_1d/` 그대로) |
| RL 신호 선택 | **검증 IC가 가장 높은 β** 하나 |
| 팩터 선택 | 단순 팩터 5개 중 **검증 IC가 가장 높은** 하나 |
| 포트폴리오 | 매일 재조정, 상위 10 롱(동일가중) / 상위 10 − 하위 10 스프레드, 수수료 0.1% (거래 가능 종목이 20개 미만이면 절반씩) |
| **돌파구 판정** | 둘 다 만족할 때만: (1) 선택된 RL 신호의 **테스트** 스프레드 비용 후 샤프 > 0, (2) RL 신호의 **테스트 IC > 선택된 팩터의 테스트 IC** |
| 이후 | 판정을 통과하면 2026-07-14 이후 미지 구간에서 **한 번만** 재확인(별도 노트북) |

스프레드(숏 포함)는 신호의 순수한 순위 정보를 재는 지표다. 현물 실계좌는 숏이 불가능하므로 실거래 가능한 형태는 "상위 10 롱 vs 등가중"으로 따로 보고한다.

- **`런타임 > 모두 실행`만 누르면 된다.** CPU로 충분, 약 10~20분(학습 없음). `crypto_horizon_retrain_1d`가 끝나 있어야 한다.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/hz_1d/crypto_signal_extraction.csv`


In [ ]:
HORIZON = '1d'   # 이 노트북은 1일 주기 전용 — 바꾸지 말 것
BUCKET_OF = {'4h': 4 * 3600, '1d': 24 * 3600}
assert HORIZON in BUCKET_OF
HZ_SEC = BUCKET_OF[HORIZON]
HZ_BARS = HZ_SEC // 300                 # 5분봉 개수 = 수익률 타겟 길이
DAY_BUCKETS = 86400 // HZ_SEC           # 하루의 버킷 수
print(f'주기 {HORIZON}: 버킷 {HZ_SEC}초 = 5분봉 {HZ_BARS}개, 하루 {DAY_BUCKETS}버킷')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = f'hz_{HORIZON}'   # 주기별 별도 폴더 — 기존 rt1006 모델·결과를 건드리지 않는다
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 1. 1일 주기 데이터·환경 불러오기 (crypto_horizon_retrain_1d와 같은 치환) + 학습된 β 풀 로드 — 학습 없음
run_cells('crypto/notebooks/crypto_hrl_earnhft_colab.ipynb', [2, 4, 6, 8, 9, 44], patch_after=2, subs={2: [('BUCKET_SEC = 1800     # 30분 버킷', 'BUCKET_SEC = HZ_SEC   # [재설계] 주기별 버킷'), ("RL_TRAIN_START = '2025-01-01'", "RL_TRAIN_START = '2021-01-01'"), ("RL_TRAIN_END   = '2025-12-31'", "RL_TRAIN_END   = '2025-09-30'   # [재설계] 검증(10~12월)을 학습에서 제외"), ("RL_EMB_H5      = CRYPTO_DIR / 'crypto_rl_embeddings.h5'", "RL_EMB_H5      = CRYPTO_DIR / f'crypto_rl_embeddings_{HORIZON}.h5'"), ('ROUTE_BUCKETS         = 48 ', 'ROUTE_BUCKETS         = DAY_BUCKETS '), ('MACRO_WINDOW           = 96 ', 'MACRO_WINDOW           = max(2 * DAY_BUCKETS, 14) ')], 6: [("df = pd.read_parquet(fp, columns=['datetime', 'RetTarget_6b'])", "df = pd.read_parquet(fp, columns=['datetime', 'close'])\n    df['RetTarget_6b'] = df['close'].shift(-HZ_BARS) / df['close'] - 1   # [재설계] 다음 버킷까지의 수익률(5분 격자는 빈틈 없음)")]})
pool = {b: PPO.load(str(MODEL_DIR / 'crypto_hrl_pool_v4' / f'agent_beta_{b}.zip')) for b in POOL_BETAS}
valid_buckets = [b for b in all_buckets if '2025-10-01' <= bucket_to_date(b) <= '2025-12-31']
test_buckets = [b for b in all_buckets if '2026-01-01' <= bucket_to_date(b) <= '2026-07-13']
print('β 풀 로드:', list(pool), f'| 검증 {len(valid_buckets)}일 · 테스트 {len(test_buckets)}일')


In [ ]:
%%time
# 2. 신호 만들기 — (a) β별 정책 종목 점수, (b) 단순 팩터. 모두 버킷 시각에 이미 알 수 있는 정보만 쓴다
from scipy.stats import spearmanr


def policy_scores(model, buckets):
    """crypto_horizon_retrain의 step_ic와 같은 경로(목표 비중을 그대로 따르는 실행)에서 매 스텝 확신도 로짓을 기록."""
    env = SparseGatedCryptoPortfolioEnv(buckets=buckets, top_k=TOP_K, **env_kwargs)
    obs, _ = env.reset(); rows, done = {}, False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        rows[env.buckets[env.t]] = np.asarray(a[:-1], dtype=np.float64)
        tw, rho, top_idx = env.action_to_weights(a)
        obs, _, done, _, _ = env.step_with_weights(tw, extra_info={'rho': rho, 'selected': top_idx.tolist()})
    return pd.DataFrame.from_dict(rows, orient='index', columns=symbols)


# 단순 팩터 — 5분봉 종가로 버킷 시각(UTC 자정) 기준 과거 수익률·변동성
fac = {k: {} for k in ['전일 수익률(모멘텀)', '전일 수익률(반전)', '1주 수익률(모멘텀)', '1주 수익률(반전)', '저변동성']}
need = set(valid_buckets) | set(test_buckets)
for sym in symbols:
    fp = FEAT_DIR / f'{sym}.parquet'
    if not fp.exists():
        continue
    s = pd.read_parquet(fp, columns=['datetime', 'close']).set_index('datetime')['close']
    m1, m7 = s / s.shift(288) - 1, s / s.shift(288 * 7) - 1
    vol = s.pct_change().rolling(288).std()
    idx = pd.to_datetime([b * BUCKET_SEC for b in sorted(need)], unit='s', utc=True)
    take = lambda x: x.reindex(idx).to_numpy()
    for name, v in [('전일 수익률(모멘텀)', take(m1)), ('전일 수익률(반전)', -take(m1)), ('1주 수익률(모멘텀)', take(m7)),
                    ('1주 수익률(반전)', -take(m7)), ('저변동성', -take(vol))]:
        fac[name][sym] = pd.Series(v, index=sorted(need))
factors = {k: pd.DataFrame(v).reindex(columns=symbols) for k, v in fac.items()}

signals = {}
for split, bk in [('valid', valid_buckets), ('test', test_buckets)]:
    for beta in POOL_BETAS:
        signals[(f'RL β={beta}', split)] = policy_scores(pool[beta], bk)
    for k, df in factors.items():
        signals[(k, split)] = df.loc[[b for b in bk if b in df.index]]
fwd = {split: pd.DataFrame({b: {s: ret_index.get((s, b), np.nan) for s in symbols} for b in bk}).T
       for split, bk in [('valid', valid_buckets), ('test', test_buckets)]}
print('신호', len(signals), '개 준비 완료')


In [ ]:
# 3. 신호 평가 — IC, 상위 10 롱(vs 등가중), 상위 10 − 하위 10 스프레드 (비용 0.1%, 일간 재조정)
K = 10
DAYS = 365


def evaluate(score, ret):
    ics, top_r, ew_r, ls_r, prev_top, prev_ls = [], [], [], [], None, None
    for b in score.index:
        if b not in ret.index:
            continue
        sc, r = score.loc[b], ret.loc[b]
        m = sc.notna() & r.notna()
        if m.sum() < 10:
            continue
        sc, r = sc[m], r[m]
        c, _ = spearmanr(sc, r)
        if not np.isnan(c):
            ics.append(c)
        k = min(K, int(m.sum()) // 2)
        order = sc.sort_values()
        top, bot = order.index[-k:], order.index[:k]
        w_top = pd.Series(1 / k, index=top)
        w_ls = pd.concat([pd.Series(1 / k, index=top), pd.Series(-1 / k, index=bot)])
        t_top = w_top.sub(prev_top, fill_value=0).abs().sum() if prev_top is not None else 1.0
        t_ls = w_ls.sub(prev_ls, fill_value=0).abs().sum() if prev_ls is not None else 2.0
        top_r.append(r[top].mean() - TC * t_top)
        ew_r.append(r.mean())
        ls_r.append(r[top].mean() - r[bot].mean() - TC * t_ls)
        prev_top, prev_ls = w_top, w_ls
    ics, top_r, ew_r, ls_r = map(np.asarray, (ics, top_r, ew_r, ls_r))
    sh = lambda x: x.mean() / (x.std() + 1e-12) * np.sqrt(DAYS)
    return {'IC': ics.mean(), 'IC t': ics.mean() / (ics.std() / np.sqrt(len(ics)) + 1e-12), '일수': len(ics),
            '스프레드 총수익(비용후)': np.prod(1 + ls_r) - 1, '스프레드 샤프(비용후)': sh(ls_r),
            '상위10 롱 총수익': np.prod(1 + top_r) - 1, '등가중 총수익': np.prod(1 + ew_r) - 1,
            '상위10 − 등가중 샤프': sh(top_r - ew_r)}


rows = []
for (name, split), score in signals.items():
    rows.append({'신호': name, '구간': split, **evaluate(score, fwd[split])})
res = pd.DataFrame(rows)


# RL 신호와 각 팩터의 순위 상관(일별 평균) — RL 신호가 단순 팩터의 재포장인지 확인
def mean_rank_corr(a, b):
    cs = []
    for d in a.index.intersection(b.index):
        x, y = a.loc[d], b.loc[d]; m = x.notna() & y.notna()
        if m.sum() >= 10:
            c, _ = spearmanr(x[m], y[m]); cs.append(c)
    return np.nanmean(cs)


v = res[res['구간'] == 'valid']
rl_pick = v[v['신호'].str.startswith('RL')].sort_values('IC').iloc[-1]['신호']
fac_pick = v[~v['신호'].str.startswith('RL')].sort_values('IC').iloc[-1]['신호']
corr = {k: mean_rank_corr(signals[(rl_pick, 'test')], signals[(k, 'test')]) for k in factors}
res.to_csv(CRYPTO_DIR / 'crypto_signal_extraction.csv', index=False, encoding='utf-8-sig')

pd.set_option('display.width', 220)
show = res.copy()
for k in ['IC']:
    show[k] = show[k].map('{:+.4f}'.format)
for k in ['IC t', '스프레드 샤프(비용후)', '상위10 − 등가중 샤프']:
    show[k] = show[k].map('{:+.2f}'.format)
for k in ['스프레드 총수익(비용후)', '상위10 롱 총수익', '등가중 총수익']:
    show[k] = show[k].map('{:+.1%}'.format)
for split in ['valid', 'test']:
    print(f'===== {split}'); print(show[show['구간'] == split].drop(columns='구간').to_string(index=False))
print(f'\n[test] 선택된 RL 신호({rl_pick})와 팩터의 평균 순위 상관:', {k: round(c, 3) for k, c in corr.items()})

t = res[res['구간'] == 'test'].set_index('신호')
c1 = t.loc[rl_pick, '스프레드 샤프(비용후)'] > 0
c2 = t.loc[rl_pick, 'IC'] > t.loc[fac_pick, 'IC']
print(f"\n[사전 등록 선택] RL 신호: {rl_pick} (검증 IC 최고) · 팩터: {fac_pick} (검증 IC 최고)")
print(f"  (1) 테스트 스프레드 비용후 샤프 {t.loc[rl_pick, '스프레드 샤프(비용후)']:+.2f} > 0 → {'충족' if c1 else '미충족'}")
print(f"  (2) 테스트 IC {t.loc[rl_pick, 'IC']:+.4f} > 팩터 {t.loc[fac_pick, 'IC']:+.4f} → {'충족' if c2 else '미충족'}")
print('  → 돌파구 판정:', '통과 — 미지 구간에서 한 번 재확인할 것' if (c1 and c2) else '통과하지 못함')
